In [1]:
import pandas as pd
import numpy as np


import torch
import torch.nn as nn
import torch.nn.functional as F


from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_percentage_error,
    mean_absolute_error
)


from src.data.data_ingestion import fetch_data
from src.data.data_transformation import split_data, transform
from src.models.lstm import LSTM
from src.utils import sliding_window, convert_array_to_tensor
from src.eval import train_validate

import warnings
warnings.filterwarnings("ignore")


# Fed Funds Series ID
SERIES_ID = "FEDFUNDS"

df = fetch_data(SERIES_ID)
df.head(10)


,FEDFUNDS,Date
0,0.80,1954-07-01
1,1.22,1954-08-01
2,1.07,1954-09-01
3,0.85,1954-10-01
4,0.83,1954-11-01
5,1.28,1954-12-01
6,1.39,1955-01-01
7,1.29,1955-02-01
8,1.35,1955-03-01
9,1.43,1955-04-01


In [2]:
# split data
df_train, df_val = split_data(df, train_size=0.80)


# transform data
df_train_scaled, df_val_scaled = transform(df_train, df_val)
df_train_scaled.shape, df_val_scaled.shape

((688, 1), (173, 1))

In [3]:
# set window size for training and testing data

WINDOW_SIZE = 20
X_train, y_train = sliding_window(df_train_scaled, WINDOW_SIZE)
X_val, y_val = sliding_window(df_val_scaled, WINDOW_SIZE)

# set device to cuda
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# convert to tensors after sliding window has been applied
X_train = convert_array_to_tensor(X_train).to(device)
y_train = convert_array_to_tensor(y_train).to(device)
X_val = convert_array_to_tensor(X_val).to(device)
y_val = convert_array_to_tensor(y_val).to(device)

In [4]:
# load in LSTM

model = LSTM(
    input_size=1,
    hidden_size=512,
    num_layers=2,
    output_size=1
)
model.to(device)

# load in optimizer and loss function
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()
epochs = 100

train_validate(
    model,
    X_train,
    y_train,
    X_val,
    y_val,
    device,
    optimizer,
    epochs,
    loss_fn
)


Epoch 1/100 | train_rmse: 0.2081 | test_rmse: 0.1280
Epoch 2/100 | train_rmse: 0.1918 | test_rmse: 0.2807
Epoch 3/100 | train_rmse: 0.1896 | test_rmse: 0.2780
Epoch 4/100 | train_rmse: 0.1719 | test_rmse: 0.2185
Epoch 5/100 | train_rmse: 0.1726 | test_rmse: 0.1777
Epoch 6/100 | train_rmse: 0.1751 | test_rmse: 0.1585
Epoch 7/100 | train_rmse: 0.1719 | test_rmse: 0.1544
Epoch 8/100 | train_rmse: 0.1627 | test_rmse: 0.1616
Epoch 9/100 | train_rmse: 0.1512 | test_rmse: 0.1792
Epoch 10/100 | train_rmse: 0.1464 | test_rmse: 0.2050
Epoch 11/100 | train_rmse: 0.1477 | test_rmse: 0.2143
Epoch 12/100 | train_rmse: 0.1265 | test_rmse: 0.1773
Epoch 13/100 | train_rmse: 0.1104 | test_rmse: 0.1268
Epoch 14/100 | train_rmse: 0.1033 | test_rmse: 0.0843
Epoch 15/100 | train_rmse: 0.0874 | test_rmse: 0.0551
Epoch 16/100 | train_rmse: 0.1529 | test_rmse: 0.0456
Epoch 17/100 | train_rmse: 0.1180 | test_rmse: 0.0580
Epoch 18/100 | train_rmse: 0.1407 | test_rmse: 0.0600
Epoch 19/100 | train_rmse: 0.1371 | t

In [5]:
# undue MinMaxScaler

scaler = MinMaxScaler()
scaler.fit(df_train)

model.eval()
with torch.no_grad():
    y_pred = model(X_val)


y_pred_np = y_pred.detach().cpu().numpy()
y_val_np = y_val.detach().cpu().numpy()

pred_rescaled = scaler.inverse_transform(y_pred_np)
actual_rescaled = scaler.inverse_transform(y_val_np)

pred_rescaled[:5], actual_rescaled[:5]

(array([[0.1844379 ],
        [0.16994493],
        [0.15649511],
        [0.14616758],
        [0.14126495]], dtype=float32),
 array([[0.09],
        [0.08],
        [0.08],
        [0.09],
        [0.08]], dtype=float32))

In [ ]:
# evaluation

r2 = r2_score(y_val_np, y_pred_np)
print(f"R2 Score: {r2:.2f}")

mape = mean_absolute_percentage_error(y_val_np, y_pred_np)
print(f"Mean Absolute Percentage Error: {mape}")


mae = mean_absolute_error(y_val_np, y_pred_np)
print(f"Mean Absolute Error: {mae:.4f}")


rmse = np.sqrt(mean_squared_error(y_val_np, y_pred_np))
print(f"Root Mean Squared Error: {rmse:.4f}")

R2 Score: 0.96
Mean Absolute Percentage Error: 397333889024.0
Mean Absolute Error: 0.0111
Root Mean Squared Error: 0.0196


In [7]:
train_len = int(len(df) * 0.80)
val_target_dates = df["Date"].iloc[train_len + WINDOW_SIZE:].reset_index(drop=True)

actual_flat = actual_rescaled.flatten()
pred_flat = pred_rescaled.flatten()


print("dates:", len(val_target_dates), "actual:", len(actual_flat), "pred:", len(pred_flat))

comparison_df = pd.DataFrame({
    "Date": val_target_dates.values,
    "Actual Fed Funds Rate": actual_flat,
    "Predicted Fed Funds Rate": pred_flat
})

print(comparison_df.head(20))
print(comparison_df.tail(20))

dates: 153 actual: 153 pred: 153
         Date  Actual Fed Funds Rate  Predicted Fed Funds Rate
0  2013-07-01                   0.09                  0.184438
1  2013-08-01                   0.08                  0.169945
2  2013-09-01                   0.08                  0.156495
3  2013-10-01                   0.09                  0.146168
4  2013-11-01                   0.08                  0.141265
5  2013-12-01                   0.09                  0.138631
6  2014-01-01                   0.07                  0.138839
7  2014-02-01                   0.07                  0.137150
8  2014-03-01                   0.08                  0.134017
9  2014-04-01                   0.09                  0.132550
10 2014-05-01                   0.09                  0.134477
11 2014-06-01                   0.10                  0.138111
12 2014-07-01                   0.09                  0.143472
13 2014-08-01                   0.09                  0.147102
14 2014-09-01         